In [3]:
# Make a graph from feci_pls where column1 is the Feci node and column2 is the PLS node (it's an undirected graph).
# Each edge will an have attribute called "direction" but they aren't defined yet
# Each node will have a "direction" attribute. Each node from column1 will have a coresponding row found (same name) 
# in the "node" column of feci.csv. In that coressponding row, the second column "direction" will be the direction attribute 
# of that node in the graph. Do this for all nodes in the graph. 
# Each node from column2 will have a coresponding row found (same name) 
# in the "node" column of pls.csv. In that coressponding row, the second column "direction" will be the direction attribute 
# of that node in the graph. Do this for all nodes in the graph. 
# 

In [16]:
import pandas as pd
import networkx as nx

# Load full edge table
feci_pls = pd.read_csv("final_edges_with_fdr.csv")  # load all columns
print("Columns in edge table:", len(feci_pls))
feci_pls = feci_pls.dropna(subset=["Pooled FDR"])
print("Columns after dropping NaN:", len(feci_pls))


# Build undirected graph
G = nx.Graph()

# Use the first two columns for edges (assuming first column is Feci, second is PLS)
feci_col = feci_pls.columns[0]
pls_col = feci_pls.columns[1]


# Add edges to the graph with direction based on "Sign" column
for i, row in feci_pls.iterrows():
    src = row[feci_col]
    tgt = row[pls_col]

    # Skip invalid/missing nodes
    if pd.isna(src) or pd.isna(tgt):
        continue  # or print a warning if you want

    src = str(src)
    tgt = str(tgt)

    # Add edge (undirected)
    G.add_edge(src, tgt)

    # Set edge "direction" attribute based on "Sign" column
    sign = row["Sign"]
    if sign == '+':
        G[src][tgt]["direction"] = 1
    elif sign == '-':
        G[src][tgt]["direction"] = -1
    else:
        print("ERROR")

print("All edges have a direction.")

Columns in edge table: 251758
Columns after dropping NaN: 40109
All edges have a direction.


In [17]:
# Load node tables
pls_nodes = pd.read_csv("pls_new.csv")
feci_nodes = pd.read_csv("feci_new.csv")

feci_node_col = feci_nodes.columns[0]
feci_dir_col = feci_nodes.columns[1]

pls_node_col = pls_nodes.columns[0]
pls_dir_col = pls_nodes.columns[1]

# Assign node directions
for u, v in G.edges():
    # Skip edges if nodes are missing
    if u not in G or v not in G:
        continue

    clean_u = u.replace("-Feci", "").replace("-PLS", "")
    clean_v = v.replace("-Feci", "").replace("-PLS", "")

    # Lookup v
    if v.endswith("-PLS"):
        row = pls_nodes[pls_nodes[pls_node_col] == clean_v]
        direction_v = row.iloc[0][pls_dir_col] if not row.empty else None
    elif v.endswith("-Feci"):
        row = feci_nodes[feci_nodes[feci_node_col] == clean_v]
        direction_v = row.iloc[0][feci_dir_col] if not row.empty else None
    else:
        direction_v = None

    # Lookup u
    if u.endswith("-PLS"):
        row = pls_nodes[pls_nodes[pls_node_col] == clean_u]
        direction_u = row.iloc[0][pls_dir_col] if not row.empty else None
    elif u.endswith("-Feci"):
        row = feci_nodes[feci_nodes[feci_node_col] == clean_u]
        direction_u = row.iloc[0][feci_dir_col] if not row.empty else None
    else:
        direction_u = None

    # Assign node attributes only if node exists
    if u in G:
        G.nodes[u]["direction"] = direction_u
    if v in G:
        G.nodes[v]["direction"] = direction_v


# ------------------------------------------------------------------------------------------------------------
nodes_to_remove = []

for node, attributes in G.nodes(data=True):
    direction = attributes.get("direction")
    if direction is None:
        nodes_to_remove.append(node)
G.remove_nodes_from(nodes_to_remove)

print(f"Removed {len(nodes_to_remove)} nodes with no direction.")
print(f"Remaining nodes: {G.number_of_nodes()}")
print(f"Remaining edges: {G.number_of_edges()}")



Removed 273 nodes with no direction.
Remaining nodes: 558
Remaining edges: 18016


In [18]:
edge_data = []

print(f"Total edges in G: {G.number_of_edges()}")  # Check number of edges

for u, v, data in G.edges(data=True):
    dir_u = G.nodes[u].get("direction")
    dir_v = G.nodes[v].get("direction")
    edge_dir = data.get("direction")

    edge_data.append({
        "node_u": u,
        "node_v": v,
        "dir_u": dir_u,
        "dir_v": dir_v,
        "edge_dir": edge_dir
    })

df_edges = pd.DataFrame(edge_data)

df_edges.to_csv("test.csv", index=False)

print("Saved", len(df_edges), "edge directions to test.csv")


Total edges in G: 18016
Saved 18016 edge directions to test.csv


In [21]:
# Collect edges to remove
edges_to_remove = []
temp = 0
for u, v, data in G.edges(data=True):
    # Get node directions
    dir_u = G.nodes[u].get("direction")
    dir_v = G.nodes[v].get("direction")
    edge_dir = data.get("direction")

    # double checking
    if dir_u is None or dir_v is None or edge_dir is None:
        print("Error")
        edges_to_remove.append((u, v))
        continue
        
    #if (temp < 10):
    #    print("u:",dir_u)
    #    print("v:",dir_v)
    #    print("edge:",edge_dir)
    
    # If product of node directions doesn't match edge direction, mark for removal
    if dir_u * dir_v != edge_dir:
        edges_to_remove.append((u, v))
    
# Remove all flagged edges
G.remove_edges_from(edges_to_remove)

print(f"Removed {len(edges_to_remove)} edges where node directions didn't match edge direction.")
print(f"Remaining edges: {G.number_of_edges()}")



Removed 7545 edges where node directions didn't match edge direction.
Remaining edges: 10471


In [22]:
# Column names
feci_col = feci_pls.columns[0]
pls_col  = feci_pls.columns[1]

# Build set of valid (Feci, PLS) pairs from graph edges
valid_pairs = set()

for u, v in G.edges():
    if isinstance(u, str) and u.endswith("-PLS"):
        valid_pairs.add((v, u))   # (Feci, PLS)
    elif isinstance(u, str) and u.endswith("-Feci"):
        valid_pairs.add((u, v))   # (Feci, PLS)

# Create a Series of row-wise (Feci, PLS) tuples
row_pairs = feci_pls[[feci_col, pls_col]].apply(tuple, axis=1)

# Filter dataframe
filtered_feci_pls = feci_pls[row_pairs.isin(valid_pairs)].copy()

print(f"Original rows: {len(feci_pls)}")
print(f"Remaining rows: {len(filtered_feci_pls)}")

filtered_feci_pls.to_csv("final.csv", index=False)
filtered_feci_pls.to_csv("../BiBC/Feci-PLS.csv", index=False)

Original rows: 40109
Remaining rows: 10471
